In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Define problem
# Always two features
# Ideal weight of first feature is always 0.5, ideal weight of second feature is configurable
# Whether features are independent is configurable, when colinear, they are the same target feature with different amounts of noise
# Noise for the first feature is always 0.1, noise for the second feature is configurable
# Note that all of the three lines are like a small version of the feature sifting setup I'm already using with how features are generated with noise coefficients and such

In [ ]:
def make_problem_data(
    n_steps,
    ideal_weights,           # optimal LMS weight per feature (scalar or list)
    *,
    feature_noise=0.0,       # noise coefficient c per feature (scalar or list)
    target_noise_std=0.1,    # std of additive Gaussian noise on the target
    shared_signal=False,     # if True, all features are noisy views of ONE signal
    seed=0,
):
    """Generate (features, target) for the feature-sifting linear-regression probe.

    Every knob that may vary across experiments is a parameter:

    - ``ideal_weights``: one optimal weight per feature -> sets the number of
      features. For independent features this is exactly the optimal LMS weight
      on that feature, so the dashed lines in the plots are the true optima.
    - ``feature_noise``: per-feature noise coefficient ``c`` in
      ``f = (1 - c) * s + c * U(-1, 1)``;  ``c = 0`` gives a clean feature.
    - ``target_noise_std``: std of the Gaussian noise added to the target.
    - ``shared_signal``: if True every feature is a noisy view of the *same*
      underlying signal (the colinear / "blocking" case); otherwise each feature
      has its own independent signal. When the signal is shared the per-feature
      optima become coupled, so ``ideal_weights`` then sets each feature's
      contribution rather than its realised optimum.

    Returns ``features`` (n_steps, n_feat) and ``y`` (n_steps,).
    """
    rng = np.random.default_rng(seed)
    ideal_weights = np.atleast_1d(np.asarray(ideal_weights, dtype=float))
    n_feat = len(ideal_weights)
    c = np.broadcast_to(np.asarray(feature_noise, dtype=float), (n_feat,))

    # underlying clean signals
    if shared_signal:
        s = rng.uniform(-1.0, 1.0, size=n_steps)
        signals = [s] * n_feat
    else:
        signals = [rng.uniform(-1.0, 1.0, size=n_steps) for _ in range(n_feat)]

    # w_ideal = a * (1 - c) * Var(s) / Var(f); Var(s) == Var(noise), so
    # Var(f) = ((1 - c)**2 + c**2) * Var(s) and the variances cancel.
    def signal_coef(w_ideal, cj):
        return 0.0 if cj >= 1.0 else w_ideal * ((1 - cj) ** 2 + cj ** 2) / (1 - cj)

    features = np.empty((n_steps, n_feat))
    y = rng.normal(0.0, target_noise_std, size=n_steps)
    for j in range(n_feat):
        feat_noise = rng.uniform(-1.0, 1.0, size=n_steps)
        features[:, j] = (1 - c[j]) * signals[j] + c[j] * feat_noise
        y = y + signal_coef(ideal_weights[j], c[j]) * signals[j]

    return features, y


In [ ]:
# First experiment:
# Independent features, ideal weight of second feature with ideal weights of 0.1, 0.3, 0.5, 0.7, and 0.9
# The second feature should also have noise of 0.1
# Both features are trained with LMS with a step-size of 0.01 for 100k steps
# I want to plot several plots, vertically stacked, each for a different ideal weight of the second feature
# Each plot should include lines for:
#   - The current weights of both features
#   - A dashed line for the ideal weight of the second feature
#   - Traces of changes to the weights with varying decay rates (0.9, 0.99, 0.999)
#   - h value from IDBD (compute even though learning happens with a constant step-size)
#   - h value from IDBD3 (compute even though learning happens with a constant step-size)


# The goal of this experiment is to figure out which, if any, of the metrics I've mentioned would be sufficient to determine when a feature should be protected
# The indicators I'm considering are traces of changes to the weights, traces to changes to the step-size (when using idbd),
#   and the h value of IDBD or IDBD3 (when its sign flips) (which can be computed even when not using IDBD).


In [ ]:
# ---------------------------------------------------------------------------
# First experiment: a single feature with clean inputs (no feature noise) and a
# noisy target, learned from a zero weight with constant-step-size LMS, while we
# *observe* candidate "maturity" indicators for its weight.
#
# Learning is plain LMS; the IDBD / IDBD3 h-traces are computed alongside with
# the same constant step-size, exactly as the project optimisers do:
#   IDBD : h <- h * max(1 - lr*f**2, 0) + lr * loss_grad
#   IDBD3: h <- h * max(1 - lr*f**2, 0) +      loss_grad
# with loss_grad = 2*error*f. They never feed back into the weights.
# (With a constant lr the two share the same decay term and differ only by the
#  factor lr, so h_idbd == lr * h_idbd3 exactly -- they coincide once normalized.)
# ---------------------------------------------------------------------------

LEARNING_RATE = 0.01
N_STEPS = 3000
IDEAL_WEIGHTS = [0.0, 0.1, 0.5, 1.0]
TRACE_DECAYS = [0.9, 0.99, 0.999]


def run_lms_with_indicators(features, y, lr=LEARNING_RATE, trace_decays=TRACE_DECAYS,
                            w_init=None):
    """LMS over `features` (n_steps, n_feat); indicators are tracked for the
    feature under study, taken to be the LAST column. `w_init` defaults to zeros
    (a fresh weight); pass a vector to start e.g. an established weight at its
    optimum in the later blocking experiments."""
    n_steps, n_feat = features.shape
    tracked = n_feat - 1
    decays = np.asarray(trace_decays, dtype=float)
    n_dec = len(decays)

    w = np.zeros(n_feat) if w_init is None else np.asarray(w_init, dtype=float)
    h_idbd = 0.0
    h_idbd3 = 0.0
    dw_tr = np.zeros(n_dec)

    w_hist = np.empty((n_steps, n_feat))
    dw_trace_hist = np.empty((n_steps, n_dec))
    h_idbd_hist = np.empty(n_steps)
    h_idbd3_hist = np.empty(n_steps)

    for t in range(n_steps):
        f = features[t]
        error = float(w @ f) - y[t]              # y_hat - y  (project convention)

        fi = f[tracked]
        loss_grad = 2.0 * error * fi             # d(error**2)/dw_tracked
        decay_term = fi * fi                     # prediction-grad squared

        # IDBD / IDBD3 h-traces (observed only, constant lr)
        keep = max(0.0, 1.0 - lr * decay_term)
        h_idbd = h_idbd * keep + lr * loss_grad
        h_idbd3 = h_idbd3 * keep + loss_grad

        # LMS weight update (constant step-size) for every weight
        dw = -lr * (2.0 * error * f)
        w = w + dw

        # EMA traces of the tracked weight's changes
        dw_tr = decays * dw_tr + (1.0 - decays) * dw[tracked]

        w_hist[t] = w
        dw_trace_hist[t] = dw_tr
        h_idbd_hist[t] = h_idbd
        h_idbd3_hist[t] = h_idbd3

    return {'w': w_hist, 'dw_trace': dw_trace_hist,
            'h_idbd': h_idbd_hist, 'h_idbd3': h_idbd3_hist}


results = {}
for w_ideal in IDEAL_WEIGHTS:
    # single clean feature (no feature noise), target noise = 0.1
    feats, targ = make_problem_data(N_STEPS, [w_ideal], feature_noise=0.0,
                                    target_noise_std=0.1, seed=0)
    results[w_ideal] = run_lms_with_indicators(feats, targ)


### Reading the plots

One panel per ideal weight $w^*$ (including $w^* = 0$ — a useless feature whose weight
should stay at zero). A single feature with clean inputs is learned from a zero weight;
all the noise lives in the target ($\sigma = 0.1$).

**Left axis (true scale, black):** the weight (solid) and its ideal value (dashed). The
grey vertical line marks where the weight first reaches 90% of its ideal — a rough
"matured" reference (omitted for $w^* = 0$).

**Right axis (linear, coloured):** the candidate maturity indicators, each normalized to
unit peak so they are comparable (raw magnitudes differ by orders of magnitude). All are
shown in the weight-change sign convention ($-h$), so each indicator is positive while
the weight grows and collapses toward (and flips around) zero once the weight matures.

The IDBD / IDBD3 $h$-traces use the project's exact recursion
$h_t = h_{t-1}\,\max(1 - \alpha f^2,\,0) + (\alpha\ \text{or}\ 1)\cdot\text{loss\_grad}$,
with $\alpha$ held constant. They share the *same* decay multiplier and their additive
terms differ only by the constant factor $\alpha$, so $h^{\text{IDBD}}_t = \alpha\,
h^{\text{IDBD3}}_t$ for all $t$ — i.e. they are exactly proportional and coincide after
normalization. They only diverge once $\alpha$ is allowed to adapt (then the decay
multipliers differ too), which is a later experiment.


In [ ]:
def _norm_peak(a):
    m = np.max(np.abs(a))
    return a / m if m > 0 else a


plot_steps = np.arange(1, N_STEPS + 1)

W_COLOR = 'black'                                    # the weight
DW_COLORS = ['tab:blue', 'tab:green', 'tab:purple']  # one per TRACE_DECAYS entry
H_IDBD3_COLOR = 'tab:red'
H_IDBD_COLOR = 'tab:orange'

fig, axes = plt.subplots(len(IDEAL_WEIGHTS), 1,
                         figsize=(11, 2.9 * len(IDEAL_WEIGHTS)), sharex=True)
ax2_first = None
for ax, w_ideal in zip(axes, IDEAL_WEIGHTS):
    res = results[w_ideal]

    # --- left axis: the weight (true scale, black) ---
    ax.plot(plot_steps, res['w'][:, -1], color=W_COLOR, lw=2.0, label='weight')
    ax.axhline(w_ideal, color=W_COLOR, ls='--', lw=1.1, alpha=0.55,
               label='ideal weight')
    ax.set_xscale('log')
    ax.set_ylabel('weight')
    ax.set_title(f'ideal weight = {w_ideal}', fontsize=10)

    # rough "matured" reference: weight first reaches 90% of its ideal (skip if 0)
    if w_ideal > 0:
        hit = np.where(res['w'][:, -1] >= 0.9 * w_ideal)[0]
        if len(hit):
            ax.axvline(hit[0] + 1, color='0.55', ls=(0, (4, 3)), lw=1.0, alpha=0.8)

    # --- right axis: normalized maturity indicators (one colour each) ---
    ax2 = ax.twinx()
    for k, d in enumerate(TRACE_DECAYS):
        ax2.plot(plot_steps, _norm_peak(res['dw_trace'][:, k]),
                 color=DW_COLORS[k], lw=1.1, alpha=0.9, label=f'dw trace (decay {d})')
    ax2.plot(plot_steps, _norm_peak(-res['h_idbd3']), color=H_IDBD3_COLOR, lw=1.6,
             label='-h (IDBD3)')
    ax2.plot(plot_steps, _norm_peak(-res['h_idbd']), color=H_IDBD_COLOR, lw=1.1,
             ls='--', label='-h (IDBD, == IDBD3 here)')
    ax2.axhline(0.0, color='0.5', lw=0.6, alpha=0.3)
    ax2.set_ylim(-1.15, 1.15)
    ax2.set_ylabel('indicators\n(norm., +1 = peak)')
    if ax2_first is None:
        ax2_first = ax2

axes[-1].set_xlabel('step (log scale)')
hL, lL = axes[0].get_legend_handles_labels()
hR, lR = ax2_first.get_legend_handles_labels()
fig.legend(hL + hR, lL + lR, loc='upper center', ncol=4, fontsize=8.5,
           bbox_to_anchor=(0.5, 1.0))
fig.tight_layout(rect=(0, 0, 1, 0.97))
plt.show()


In [ ]:
# Later experiments will look at:
#   - Learning while optimizing the step-size
#   - Learning with different amounts of noise in the incumbent feature
#   - Learning with the estavblished feature blocking the incumbent feature

# These don't need to be implemented for now